In [1]:
!pip install ultralytics

import os
import shutil
import yaml
import time
import numpy as np
import cv2
import torch
import glob
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
from ultralytics import YOLO

# Paths
SOURCE_IMG_DIR = "/content/data/data/images"
SOURCE_MASK_DIR = "/content/data/data/masks"
YOLO_ROOT = "/content/yolo_dataset"

# --- The "Proven" Conversion Logic ---
def mask_to_yolo_polygon(mask_path):
    # Read mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: return []
    
    # FIX: Handle 0-1 masks (scale to 0-255)
    if np.max(mask) <= 1:
        mask = (mask * 255).astype(np.uint8)
    
    # Threshold
    _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    h, w = mask.shape
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 20: # Filter noise
            polygon = []
            for point in cnt:
                x, y = point[0]
                polygon.append(x / w)
                polygon.append(y / h)
            if len(polygon) > 4:
                polygons.append(polygon)
    return polygons

def regenerate_dataset(seed):
    """Regenerates the dataset split for a specific random seed"""
    
    # 1. Clean Directories
    if os.path.exists(YOLO_ROOT): shutil.rmtree(YOLO_ROOT)
    for split in ['train', 'test']:
        os.makedirs(f"{YOLO_ROOT}/{split}/images", exist_ok=True)
        os.makedirs(f"{YOLO_ROOT}/{split}/labels", exist_ok=True)
    
    # 2. Split Data
    all_files = sorted([f for f in os.listdir(SOURCE_IMG_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=seed)
    
    # 3. Process
    for split, files in [('train', train_files), ('test', test_files)]:
        for f in files:
            # Copy Image
            shutil.copy(os.path.join(SOURCE_IMG_DIR, f), os.path.join(YOLO_ROOT, split, 'images', f))
            
            # Find Mask (Try multiple extensions)
            base_name = os.path.splitext(f)[0]
            label_path = os.path.join(YOLO_ROOT, split, 'labels', base_name + ".txt")
            
            mask_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                try_path = os.path.join(SOURCE_MASK_DIR, base_name + ext)
                if os.path.exists(try_path):
                    mask_path = try_path
                    break
            
            # Create Label
            if mask_path:
                polygons = mask_to_yolo_polygon(mask_path)
                with open(label_path, 'w') as out:
                    if polygons:
                        for poly in polygons:
                            out.write("0 " + " ".join(map(str, poly)) + "\n")
                    # If polygons empty, file stays empty (background)
            else:
                # No mask found -> Empty file (background)
                open(label_path, 'w').close()

    # 4. Create YAML
    data_config = {
        'path': YOLO_ROOT, 'train': 'train/images', 'val': 'test/images', 'names': {0: 'polyp'}
    }
    with open(f"{YOLO_ROOT}/data.yaml", 'w') as f:
        yaml.dump(data_config, f)
        
    return test_files # Return test set for evaluation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# 2. Download Data
import gdown
file_id = '1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

if os.path.exists(output) and not os.path.exists("/content/data/data"):
    print("Unzipping...")
    !unzip -o -q data.zip -d /content/data
    print("✅ Data ready.")
else:
    print("✅ Data already extracted.")

Downloading...
From: https://drive.google.com/uc?id=1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI
To: /content/data.zip
100%|██████████| 18.0M/18.0M [00:00<00:00, 67.6MB/s]


Unzipping...
✅ Data ready.


In [3]:
def run_full_cycle(seed, epochs=30):
    print(f"\n{'='*40}")
    print(f"🚀 STARTING RUN WITH SEED: {seed}")
    print(f"{'='*40}")
    
    # 1. Regenerate Data (Fixes the split for this seed)
    test_files = regenerate_dataset(seed)
    
    # 2. FORCE CLEAR CACHE (Crucial!)
    for split in ['train', 'test']:
        cache = f"{YOLO_ROOT}/{split}/labels.cache"
        if os.path.exists(cache): os.remove(cache)
        
    # 3. Train
    model = YOLO('yolo11m-seg.pt')
    model.train(
        data=f"{YOLO_ROOT}/data.yaml",
        epochs=epochs,
        imgsz=256,
        batch=16,
        project="/content/yolo_results",
        name=f"run_{seed}",
        augment=True,
        exist_ok=True,
        verbose=False
    )
    
    # 4. Evaluate
    print("   Running Evaluation...")
    scores = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}
    
    # Predict on test set
    results = model.predict(source=f"{YOLO_ROOT}/test/images", imgsz=256, conf=0.25, verbose=False, retina_masks=True)
    
    for result in results:
        # Match Prediction to GT
        filename = os.path.basename(result.path)
        base_name = os.path.splitext(filename)[0]
        
        # Find GT Mask
        gt_path = None
        for ext in ['.png', '.jpg']:
            p = os.path.join(SOURCE_MASK_DIR, base_name + ext)
            if os.path.exists(p): gt_path = p; break
            
        if not gt_path: continue

        # Process Pred
        h, w = result.orig_shape
        pred_mask = np.zeros((h, w), dtype=np.uint8)
        if result.masks:
            for m in result.masks.data:
                m_np = m.cpu().numpy()
                m_resized = cv2.resize(m_np, (w, h), interpolation=cv2.INTER_NEAREST)
                pred_mask = np.maximum(pred_mask, m_resized)
        pred_bin = (pred_mask > 0.5).astype(np.uint8).flatten()
        
        # Process GT
        gt = np.array(Image.open(gt_path).convert("L"))
        gt_bin = (gt > 127).astype(np.uint8).flatten()
        
        # Metrics
        scores["dice"].append(f1_score(gt_bin, pred_bin, zero_division=1))
        scores["iou"].append(jaccard_score(gt_bin, pred_bin, zero_division=1))
        scores["prec"].append(precision_score(gt_bin, pred_bin, zero_division=1))
        scores["rec"].append(recall_score(gt_bin, pred_bin, zero_division=1))
        scores["acc"].append(accuracy_score(gt_bin, pred_bin))
        
    avg_scores = {k: np.mean(v) for k, v in scores.items()}
    print(f"   Done. Dice: {avg_scores['dice']:.4f} | IoU: {avg_scores['iou']:.4f}")
    
    return avg_scores, model

In [4]:
SEEDS = [42, 0, 123, 7, 2026]
EPOCHS = 50 # Adjust if needed

final_stats = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}
last_trained_model = None

for seed in SEEDS:
    metrics, model = run_full_cycle(seed, epochs=EPOCHS)
    last_trained_model = model
    
    for k, v in metrics.items():
        final_stats[k].append(v)

# Report
print("\n" + "="*50)
print("✅ FINAL MULTI-RUN REPORT (YOLOv11)")
print("="*50)
print(f"{'Metric':<10} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 35)

for k, v in final_stats.items():
    print(f"{k.upper():<10} | {np.mean(v):.4f}     | {np.std(v):.4f}")


🚀 STARTING RUN WITH SEED: 42
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=run_42, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [5]:
import time

ITERATIONS = 100
WARMUP = 20
IMG_SIZE = 256
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"⏱️ Benchmarking Inference Speed on {device_name}...")

# Dummy Input
dummy_img = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

# Warmup
print(f"   Warming up ({WARMUP} runs)...")
for _ in range(WARMUP):
    _ = last_trained_model.predict(dummy_img, verbose=False, imgsz=IMG_SIZE)

# Measure
timings = []
print(f"   Measuring ({ITERATIONS} runs)...")
for _ in range(ITERATIONS):
    start = time.time()
    _ = last_trained_model.predict(dummy_img, verbose=False, imgsz=IMG_SIZE)
    end = time.time()
    timings.append((end - start) * 1000)

avg_ms = np.mean(timings)
std_ms = np.std(timings)
fps = 1000 / avg_ms

print("\n" + "="*30)
print("🚀 FINAL SPEED RESULTS")
print("="*30)
print(f"Avg Time: {avg_ms:.2f} ms ± {std_ms:.2f} ms")
print(f"FPS:      {fps:.2f} frames/sec")
print("="*30)

⏱️ Benchmarking Inference Speed on Tesla T4...
   Warming up (20 runs)...
   Measuring (100 runs)...

🚀 FINAL SPEED RESULTS
Avg Time: 15.47 ms ± 3.05 ms
FPS:      64.65 frames/sec
